# Sprint 2 · Track and explain controlled experiments

Linden Facilities needs 24 hourly energy-demand predictions as one file within 60 seconds. Work locally in the same clone and Python 3.12 project environment. These notebook sections follow the Campus journey in order and import the same shared core modules used by the project.

Follow [repository setup](https://github.com/richhiey/ml-llm-ops/blob/main/docs/setup.md), select its environment as the notebook kernel, and run sections in order. Read the task and make your prediction before executing a probe. Cells tagged `student-work` are your independent attempts; finish those decisions before running a later cell that consumes them. Guided outputs alone do not complete an independent task.

Keep one set of work and reuse completed runs. A fresh kernel reopens `work/project.json`; it does not silently choose another model. The core plotting functions display actual evidence and save SVG/PNG copies under `work/figures`. Exported Campus figures are verified reference examples; your custom experiment can produce different observations.


In [ ]:
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
from pathlib import Path
import sys, json
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Use the project's Python 3.12 environment from the local setup guide.")
from linden_mlops import (Config, Workspace, ingest, preprocess, train, evaluate,
    run_pipeline, export_runs, verify_summary, register, approve, activate,
    batch_predict, reproduce)
from linden_mlops.lifecycle import ROOT, MODEL_NAME, code_revision
from linden_mlops.project import Project
from linden_mlops.dev import course_cli, project_tests, project_python, prepare_case
from linden_mlops.course_evidence import prepare
from linden_mlops.visuals import (
    plot_contract, plot_experiments, plot_identity, plot_recovery, save_figure)
from IPython.display import display
import pandas as pd
import numpy as np
progress = Project()
project = PACKAGE = ROOT
state = progress.work
ws = Workspace(progress.tracking)
figures = state / 'figures'
print("Package source:", ROOT.name)
print("Saved decision fields:", sorted(progress.data))
print("Source identity:", code_revision())


## A-S2-01 · Why ad hoc tracking breaks down at scale

Linden's planner sees “run 4: good”. Which source, data and configuration produced it? A run ID names one execution; parameters describe what ran, metrics describe measured behavior, and artifacts retain outputs. Logging them together makes comparison inspectable. A filename or rounded score cannot replace that link.

Use the Campus case to identify what is missing from an ad hoc note. No extra execution is needed.

### Learning outcome

Identify 3+ failure modes of tracking experiments manually (spreadsheets, notebook print statements) once a team runs many models, and explain why systematic tracking matters for auditability.

[Campus explanation, task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-2/01-why-ad-hoc-tracking-breaks-down-at-scale/lesson.md).


## A-S2-02 · Controls, confounders, and decision metrics

The decision metric is validation `rmse_kwh` (lower is better), tied to historical error in the planning brief's units. MAE helps diagnose error, and autologged training scores describe fit on different rows. They do not replace held-out RMSE. Keep data bytes, split, feature order, training-only scaler fit, solver, guard and source fixed; vary `alpha` only. The reserved final rows remain closed until selection is fixed.

| Comparison | Interpretation |
|---|---|
| Same controls, different alpha | Controlled regularisation comparison |
| Different partition or source | Different question; exclude from this ranking |
| Same model, logging on/off | Logging inspection; not another alpha trial |

Predict why a lower training error does not by itself justify promotion.

### Learning outcome

Explain why a run comparison is invalid when dataset split, evaluation conditions, or multiple uncontrolled factors change, and distinguish a primary decision metric from diagnostic metrics.

[Campus explanation, task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-2/02-controls-confounders-and-decision-metrics/lesson.md).


## A-S2-03 · Building the five-run experiment matrix

Edit `configs/experiment.json` in this repository. Choose five distinct positive alpha values spanning weaker and stronger regularisation, including `1` as the comparison anchor. The supplied values are a starting range, not a required winning answer. In one short note state spacing rationale, fixed controls, why validation RMSE is primary, and an excluded example using different evaluation conditions. Do not execute that example or open the final holdout.

The next cell records your saved config. A completed matrix cannot silently become a different plan; use a deliberate new work directory only when you intend a new experiment.

### Learning outcome

Define a five-run experiment matrix for the provided training and inference pipeline starter that holds the dataset split and evaluation conditions constant, names the factor varied in each comparison, selects a primary metric tied to the inference brief, and labels any run that is not directly comparable.

[Campus explanation, task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-2/03-building-the-five-run-experiment-matrix/lesson.md).


In [ ]:
plan = json.loads((ROOT / 'configs/experiment.json').read_text())
assert (plan['primary_metric'], plan['direction'], plan['partition']) == ('rmse_kwh', 'minimize', 'validation')
print('Declared alphas:', progress.plan(plan['alphas']))

## A-S2-04 · Revisiting MLflow in a pipeline context

`Workspace.run` opens a tracked execution, calls the pipeline, explicitly logs held-out metrics, and stores the fitted model and recovery recipe. Autologging adds estimator metadata during fitting. Each ID below must have one guarded validation configuration. The cell reopens a completed matrix rather than training it again.

### Learning outcome

Use the provided MLflow setup to log parameters, metrics, and artifacts automatically as part of the cumulative pipeline, building on basic experiment logging from T4 (AI track).

[Campus explanation, task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-2/04-revisiting-mlflow-in-a-pipeline-context/lesson.md).


In [ ]:
if not progress.data.get('matrix_run_ids'):
    records = [ws.run(Config(alpha=a, range_guard=True, autolog=True), name=f'alpha-{a}')
               for a in progress.require('planned_alphas')]
    progress.record_matrix(ws, records)
rows = progress.matrix(ws)
rows.to_csv(state / 'runs.csv', index=False)
display(rows.sort_values('alpha'))
print('Saved IDs:', progress.require('matrix_run_ids'))

Inspect one actual record next. A parameter answers what ran; `rmse_kwh` answers held-out prediction error; `reproduction/manifest.json` connects recipe and environment. The model URI identifies the fitted artifact. Look inside `src/linden_mlops/lifecycle.py` to find the calls that logged these fields.

In [ ]:
sample_id = progress.require('matrix_run_ids')[0]
raw = ws.client.get_run(sample_id)
print('Run:', sample_id)
print('Parameters:', raw.data.params)
print('Metrics:', raw.data.metrics)
print('Artifacts:', [a.path for a in ws.client.list_artifacts(sample_id)])
print('Recovery files:', [a.path for a in ws.client.list_artifacts(sample_id, 'reproduction')])
print('Local UI command: uv run linden-mlops ui --state', ws.path)

## A-S2-05 · Comparing multiple runs systematically

Open the printed command in a separate local terminal, then `http://127.0.0.1:5050`. Find `linden-demand`, select the five saved IDs, open their comparison and display `alpha` and `rmse_kwh`. Compare controls in the raw rows and identify the minimum. Use the UI from this local project environment.

Choose the actual ID yourself. In the cell below call `progress.select(ws, 'your actual ID')` after checking the UI and raw table. It validates the declared numeric rule and saves the full configuration for registry/recovery. It does not prove deployment cost or future accuracy.

### Learning outcome

Use the MLflow UI to compare 5+ experiment runs against each other and identify the best-performing configuration for a given metric.

[Campus explanation, task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-2/05-comparing-multiple-runs-systematically/lesson.md).


In [ ]:
# Your choice: progress.select(ws, 'ACTUAL_RUN_ID')
# Explain the comparable controls and why this metric supports your choice.

In [ ]:
selection = progress.selected()
print('Retained selection:', selection)
(state / 'selected-config.json').write_text(json.dumps(selection['config'], indent=2) + '\n')

### Read the comparable metric plot

Both vertical axes are in kWh, but RMSE and MAE answer different error questions. Alpha uses a logarithmic horizontal axis; each point is an observed run, with no implied intermediate fit. The highlighted winner follows the raw validation RMSE rule, not a training diagnostic or final score. A small winning margin is not evidence of statistical significance.


In [ ]:
experiment_figure = plot_experiments(rows)
display(experiment_figure)
print(experiment_figure.linden_data)
experiment_paths = save_figure(experiment_figure, figures / 'experiment-comparison__01__en')


## A-S2-06 · Autologging and reducing manual tracking overhead

Before making another run, predict which parameter/metric fields autologging should add and which the wrapper must still log explicitly. Reuse the autologged `alpha=1` anchor; configure one otherwise identical manual run yourself, then retain its ID. Compare raw key sets, held-out metric agreement and model artifact counts. Explain `log_models=False` from the actual hook call. Use the public API and the Campus hints; do not run another five-run matrix.

### Learning outcome

Configure autologging for the pipeline and explain the tradeoff between autologging convenience and control over what gets logged.

[Campus task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-2/06-autologging-and-reducing-manual-tracking-overhead/lesson.md).


In [ ]:
# Your attempt: create one Config(alpha=1, range_guard=True, autolog=False) run.
# Keep the existing autologged anchor; save both IDs in progress.data["logging_pair"] via progress.save.
# Compare run.data.params, run.data.metrics and artifacts through ws.client.
# Predict fields first and verify held-out RMSE agrees within 1e-6 kWh.


### Compare after configuring the logging pair

<details><summary>Open after your own logging comparison</summary>

**Result you should get:** the autologged record contains an estimator field such as `ridge__alpha` absent from the manual record. Both keep explicit validation `rmse_kwh` and one explicitly logged model. Validation RMSE agrees within `1e-6` kWh.

**How:** compare sets of actual keys, not just flags. Enable the hook before fitting; it cannot retroactively observe a completed fit. A training diagnostic does not replace the declared held-out metric. Inspect the Campus feedback if predictions differ before attributing the difference to logging.

</details>


## A-S2-07 · Validating an assistant-generated experiment summary

Give your institution-approved assistant the five comparable raw rows and this request: “Recommend a run using lower validation rmse_kwh. Name the run ID, explain comparable controls, and separate measurements from claims needing other evidence.” Keep its actual response in your working notes. No API key or integration is required by this project.

Use `verify_summary(rows, proposed_id)` for the numeric recommendation. Break its prose into claims and mark each supported, inaccurate or unsupported with the relevant raw field. A correct ranking cannot establish cost, future generalisation or production readiness. If the response is accurate, record that; do not invent an error.

First observe a deliberately wrong numeric recommendation below. Expect `False`. The function also rejects incompatible evidence, which is different from a poor ranking.

### Learning outcome

Direct an approved assistant to generate a summary comparison across logged experiment runs, validate every recommendation against the raw MLflow data and the declared primary metric, and flag any inaccuracy or invalid comparison.

[Campus explanation, task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-2/07-validating-an-assistant-generated-experiment-summary/lesson.md).


In [ ]:
worst_id = rows.sort_values('rmse_kwh').iloc[-1].run_id
wrong = verify_summary(rows, worst_id)
print('Deliberately wrong advice:', wrong)
assert not wrong['recommendation_matches_metric']

In [ ]:
# Add the actual response's proposed ID and run verify_summary(rows, proposed_id).
# Keep one claim -> raw field -> supported/inaccurate/unsupported table.
# Qualify only actual issues; the response may be fully accurate.

## A-S2-08 · Experiment tracking deliverable

Harbor Facilities supplies six observation labels and a confident recommendation assembled from your actual records. Prepare the packet once below. Inspect `observations.json` and `client-note.txt` before deciding how many independent configurations it contains. Which observations may enter the validation ranking? What can the retained evidence establish about cost or readiness?

Preserve your actual approved-assistant prompt/input/response and claim table from the prior activity; the supplied stakeholder note is separate authored exercise text. Reuse the matrix, logging pair and selection. Keep the actual final holdout unopened.

### Learning outcome

Deliverable: at least 5 experiment configurations logged and compared in MLflow, an autologging setup, and an AI-assisted summary recommending which configuration to promote - with the summary validated against raw data.

[Campus task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-2/08-experiment-tracking-deliverable/lesson.md).


In [ ]:
comparison_packet = state / 'harbor-comparison'
if not comparison_packet.exists():
    prepare('comparison', comparison_packet, work_dir=state)
observations = json.loads((comparison_packet / 'observations.json').read_text())
display(pd.DataFrame(observations['observations']).drop(columns='controls'))
print((comparison_packet / 'client-note.txt').read_text())


In [ ]:
# Your attempt: group observations by actual run_id and inspect metric/evaluation basis.
# Audit the note against raw fields, retain eligible validation IDs, and justify the rank.
# Explain cost/readiness evidence that the table cannot supply. Do not open final rows.


### Compare after auditing Harbor's recommendation

<details><summary>Open after your observation/run audit</summary>

**Result you should get:** six labels describe five actual matrix run IDs. The repeated run's training-root-mean-squared-error observation is not another validation candidate. Rank only the five comparable validation `rmse_kwh` observations and retain your actual selected ID. No cost or readiness conclusion follows from that ranking.

**How:** separate measurement identity, execution identity and evaluation basis. Cross-check values/controls against raw MLflow records. Your actual assistant record remains a separate evidence item; a prepared client note cannot substitute for the required request.

</details>
